# BreastDCEDL_ISPY2



In [1]:
from sklearn.metrics import classification_report,auc,roc_auc_score
from PIL import Image
import time
from pathlib import Path


import os
import numpy as np
import pandas as pd
from PIL import Image


import warnings
warnings.filterwarnings('ignore', '.*do not.*', )
warnings.warn('Do not show this message')

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from glob import glob
#from skimage import io
from sklearn.utils import shuffle

#from nipype.interfaces.ants import N4BiasFieldCorrection
import sys
import os
import ast
import re
import nibabel as nib
import warnings
warnings.filterwarnings('ignore')

In [2]:
def show_n_images(
    imgs,
    cmap="gray",
    titles=None,
    enlarge=4,
    mtitle=None,
    cut=False,
    axis_off=True,
    fontsize=15,
    cb=False,
):
    """Visualise *n* images side-by-side."""
    _ = plt.figure(figsize=(enlarge * len(imgs), enlarge * 2))
    for i, im in enumerate(imgs, 1):
        ax = plt.subplot(1, len(imgs), i)
        if cb and len(np.unique(im)) > 5:
            im = cont_br(im)
        ax.imshow(im[50:290, 75:450] if cut else im, cmap=cmap, origin="lower")
        if titles is not None:
            ax.set_title(titles[i - 1], fontsize=fontsize)
        if axis_off:
            ax.axis("off")
    if mtitle:
        plt.suptitle(mtitle, fontsize=fontsize + 2)
    plt.tight_layout(); plt.show()


In [ ]:

def read_nifti(path: str) -> np.ndarray:
    return nib.load(path).get_fdata()

def _last_int(path):
    m = re.search(r"(\d+)\.nii\.gz$", path)
    return int(m.group(1)) if m else -1


def get_sorted_nifti_acquisitions(pid):

    for part in os.listdir(dce_path):
        path = os.path.join(dce_path, part, pid)
        if os.path.isdir(path):
            files = sorted(f for f in os.listdir(path) if f.endswith('.nii.gz'))
            if deb:
                print("Found:", path)
                print("Files:", files)
            return [read_nifti(os.path.join(path, f)) for f in files]
    return []



def get_nifti_mask(pid: str):

    mpath = mask_path
    if deb: print(mpath)

    ff = [x for x in os.listdir(mpath) if pid in x]

    if deb:print(ff)
    if len(ff)==0: return None

    f=ff[0]

    x = read_nifti(os.path.join(mpath, f))

    return x


In [4]:
def minmax(m):
    if m.max()==0:
        return m
    m = (m-m.min())/(m.max()-m.min())
    return m

def to_rgb(a,b,c):

    stack_axis = a.ndim

    # Stack along the next axis
    im = np.stack([minmax(a), minmax(b), minmax(c)], axis=stack_axis)

    return im

# Download data from zenodo


## Download dce data to dce directory
* follow this example to all other dce data files

In [ ]:
# set to base of your local nifti data
data_path="."

In [ ]:
# Set up your local nifti path's
import os
dce_path= os.path.join(data_path,"dce")

mask_path=os.path.join(data_path,"BreastDCEDL_ISPY2_mask")

In [ ]:
df=pd.read_csv('BreastDCEDL_ISPY2_metadata.csv')
df.info()

In [ ]:
df.test.value_counts(dropna=False)

In [ ]:
df.pCR.value_counts(dropna=False)

In [ ]:
pd.crosstab(df.test,df.pCR, dropna=False)

In [ ]:
pd.crosstab(df.test,df.HR, dropna=False)

In [ ]:
pd.crosstab(df.test,df.HER2, dropna=False)

In [ ]:
os.listdir(mask_path)[:4]

## I-SPY-2 example

In [ ]:
df[df.pid.isin(['ISPY2-913225'])]

In [ ]:
deb=0

In [ ]:

    pid='ISPY2-913225'

    row = df[df.pid==pid].iloc[0]
    smask=int(row['mask_start'])
    emask = int(row['mask_end'])
    pmask = (smask+emask)//2
    nz=int(row['n_z'])
    nxy=int(row['n_xy'])


    d=get_sorted_nifti_acquisitions(pid)
    m = get_nifti_mask(pid)

    print('dce',nz,nxy,d[0].shape, 'mask: ', smask,emask,  m.shape)

    show_n_images([d[0][k] for k in range(0,nz,5)])
    show_n_images([d[1][k] for k in range(0,nz,5)])
    show_n_images([d[2][k] for k in range(0,nz,5)])
    show_n_images([m[k] for k in range(0,nz,5)])
    show_n_images([to_rgb(d[0][k],d[1][k], d[2][k]) for k in range(0,nz,5)])
    show_n_images([to_rgb(d[0][k],d[1][k], m[k]) for k in range(0,nz,5)])
    show_n_images([to_rgb(d[0][k],d[1][k], d[2][k]) for k in [pmask-3,pmask,pmask+3]])
    show_n_images([to_rgb(d[0][k],d[1][k], m[k]) for k in [pmask-3,pmask,pmask+3]])
    show_n_images([ m[k] for k in [pmask-3,pmask,pmask+3]])




In [ ]:

    pid='ISPY2-955953'
    row = df[df.pid==pid].iloc[0]
    smask=int(row['mask_start'])
    emask = int(row['mask_end'])
    pmask = (smask+emask)//2
    nz=int(row['n_z'])
    nxy=int(row['n_xy'])
    print('mask: ', smask,emask)

    d=get_sorted_nifti_acquisitions(pid)
    m = get_nifti_mask(pid)

    im=to_rgb(d[0],d[1], d[2])
    show_n_images([to_rgb(d[0][k],d[1][k], d[2][k]) for k in [pmask-3,pmask,pmask+3]])
    show_n_images([to_rgb(d[0][k],d[1][k], m[k]) for k in [pmask-3,pmask,pmask+3]])
    show_n_images([ m[k] for k in [pmask-3,pmask,pmask+3]])

In [ ]:

    pid='ISPY2-940186'
    row = df[df.pid==pid].iloc[0]
    smask=int(row['mask_start'])
    emask = int(row['mask_end'])
    pmask = (smask+emask)//2
    nz=int(row['n_z'])
    nxy=int(row['n_xy'])
    print('mask: ', smask,emask)

    d=get_sorted_nifti_acquisitions(pid)
    m = get_nifti_mask(pid)

    im=to_rgb(d[0],d[1], d[2])
    show_n_images([to_rgb(d[0][k],d[1][k], d[2][k]) for k in [pmask-3,pmask,pmask+3]])
    show_n_images([to_rgb(d[0][k],d[1][k], m[k]) for k in [pmask-3,pmask,pmask+3]])
    show_n_images([ m[k] for k in [pmask-3,pmask,pmask+3]])